# Fine-tuning a pre-trained CNN for MNIST and saving a `.pth` model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY_NAME/blob/main/train_mnist_resnet18.ipynb)

This notebook fine-tunes a pre-trained ResNet18 convolutional neural network on the MNIST handwritten digit dataset for a small number of epochs. It then saves the trained weights as `mnist_resnet18.pth`, which is used by the Gradio app deployed on Hugging Face Spaces.

> Before submission, replace `YOUR_GITHUB_USERNAME` and `YOUR_REPOSITORY_NAME` in the Colab badge with your real GitHub username and repository name.

In [ ]:
# In Colab, uncomment this line if needed:
# !pip install torch torchvision matplotlib tqdm

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Load and prepare the MNIST data

ResNet18 was originally trained on ImageNet images. To adapt MNIST to this architecture, the 28×28 grayscale digits are resized to 224×224 and kept as one-channel images. The first convolutional layer of ResNet18 will be adapted from 3 input channels to 1 input channel.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1307], std=[0.3081]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1307], std=[0.3081]),
])

data_dir = "./data"
full_train_dataset = datasets.MNIST(root=data_dir, train=True, download=True, transform=train_transform)
test_dataset = datasets.MNIST(root=data_dir, train=False, download=True, transform=test_transform)

train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}")

## 2. Build the pre-trained CNN

The model starts from a pre-trained ResNet18. Its first layer is adapted for grayscale images by averaging the original RGB filters, and the final fully connected layer is replaced with a 10-class classifier for digits 0–9.

In [ ]:
def build_model():
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)

    # Adapt the first convolutional layer from RGB input to grayscale input.
    old_conv = model.conv1
    new_conv = nn.Conv2d(
        in_channels=1,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False,
    )
    with torch.no_grad():
        new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
    model.conv1 = new_conv

    # Replace the classifier head for MNIST's 10 classes.
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model

model = build_model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print(model.fc)

## 3. Train/fine-tune for a couple of epochs

For the assignment, two epochs are enough to demonstrate the full training and deployment pipeline. You can increase `num_epochs` if you want a slightly better model.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

num_epochs = 2
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f} | "
        f"Val loss: {val_loss:.4f}, Val acc: {val_acc:.4f}"
    )

## 4. Test the model

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

## 5. Save the model weights as a `.pth` file

This is the important deployment step. The Hugging Face Space loads this file in `app.py`.

In [ ]:
model_path = "mnist_resnet18.pth"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "architecture": "resnet18_grayscale_mnist",
    "num_classes": 10,
    "normalization_mean": [0.1307],
    "normalization_std": [0.3081],
    "input_size": [224, 224],
    "epochs": num_epochs,
    "test_accuracy": test_acc,
}

torch.save(checkpoint, model_path)
print(f"Saved model to: {model_path}")
print(f"File size: {os.path.getsize(model_path) / (1024*1024):.2f} MB")

## 6. Download the `.pth` file from Colab

After running the notebook in Colab, download `mnist_resnet18.pth` and upload it to the root folder of your Hugging Face Space together with `app.py`, `requirements.txt`, and the `examples/` folder.

In [ ]:
# Run this cell in Google Colab to download the trained model file.
try:
    from google.colab import files
    files.download("mnist_resnet18.pth")
except Exception as e:
    print("This download cell only works in Google Colab.")
    print(e)

## 7. Expected deployment files

Your Hugging Face Space should contain:

```text
app.py
requirements.txt
mnist_resnet18.pth
examples/
    digit_0.png
    digit_2.png
    digit_5.png
    digit_8.png
```

The Gradio app loads the saved `.pth` file, preprocesses the uploaded or drawn digit, and returns the estimated probability for each class from 0 to 9.